# Cat vs Dog Classifier - Overfitting Solution with Basic CNN

This notebook is a **separate overfitting-focused version** of the Cat vs Dog Classifier.

We will use:

- Kaggle API through **Google Colab Secrets**
- Microsoft Cats vs Dogs dataset
- Basic CNN model
- Data augmentation
- Dropout
- Batch Normalization
- L2 Regularization
- EarlyStopping
- ReduceLROnPlateau
- ModelCheckpoint
- Accuracy/Loss graphs
- Single image prediction

The goal is to reduce overfitting while still using a **basic CNN model**, not transfer learning.


## What is Overfitting?

Overfitting happens when a model performs very well on training data but poorly on validation/test data.

Example:

| Training Accuracy | Validation Accuracy | Meaning |
|---|---|---|
| 97% | 72% | Model memorized training images |
| 85% | 83% | Model generalizes better |

In image classification, overfitting happens when the CNN learns image-specific patterns instead of general cat/dog features.


## How We Will Reduce Overfitting

We will use the following techniques:

| Technique | Purpose |
|---|---|
| Data Augmentation | Creates variations of images during training |
| Dropout | Randomly disables neurons during training |
| Batch Normalization | Stabilizes learning |
| L2 Regularization | Penalizes overly complex weights |
| GlobalAveragePooling2D | Reduces model parameters compared to Flatten |
| EarlyStopping | Stops training when validation loss stops improving |
| ReduceLROnPlateau | Reduces learning rate when improvement slows |
| ModelCheckpoint | Saves the best model automatically |


# Step 1: Install Kaggle API

In [ ]:
!pip install -q --upgrade kaggle

# Step 2: Load Kaggle Credentials from Colab Secrets

Before running this cell, add these two secrets in Colab:

1. `KAGGLE_USERNAME`
2. `KAGGLE_KEY`

Go to the left sidebar in Colab → click the **key icon** → add both secrets.


In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

print("Kaggle credentials loaded successfully.")

# Step 3: Download Dataset from Kaggle

We will use the Microsoft Cats vs Dogs dataset.


In [ ]:
!mkdir -p /content/data
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data

# Step 4: Unzip Dataset

In [ ]:
import zipfile
import os

zip_path = "/content/data/microsoft-catsvsdogs-dataset.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")

# Step 5: Check Dataset Structure

In [ ]:
base_dir = "/content/data/PetImages"

print("Available folders:", os.listdir(base_dir))
print("Cat images:", len(os.listdir(os.path.join(base_dir, "Cat"))))
print("Dog images:", len(os.listdir(os.path.join(base_dir, "Dog"))))

# Step 6: Remove Corrupted Images

This Kaggle dataset contains a few corrupted images.  
If we do not remove them, training may stop with an error.


In [ ]:
from PIL import Image

classes = ["Cat", "Dog"]
removed = 0

for class_name in classes:
    folder_path = os.path.join(base_dir, class_name)

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)

        try:
            img = Image.open(img_path)
            img.verify()
        except Exception:
            os.remove(img_path)
            removed += 1

print("Corrupted images removed:", removed)

# Step 7: Create Training and Validation Data with Augmentation

## Why Data Augmentation Helps

Data augmentation creates modified versions of training images:

- rotated images
- shifted images
- zoomed images
- flipped images

This helps the model learn general cat/dog features instead of memorizing images.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 150
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    # Overfitting reduction through augmentation
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    base_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

print("Class labels:", train_data.class_indices)

# Step 8: Visualize Augmented Images

This helps us understand how the model sees transformed training images.


In [ ]:
import matplotlib.pyplot as plt

images, labels = next(train_data)

plt.figure(figsize=(10, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title("Dog" if labels[i] == 1 else "Cat")
    plt.axis("off")

plt.tight_layout()
plt.show()

# Step 9: Build Improved Basic CNN Model

This is still a **basic CNN**, but it is designed to reduce overfitting.

Important changes:

1. `BatchNormalization()` after convolution layers  
2. `Dropout()` after pooling layers  
3. `L2 regularization` in Conv2D and Dense layers  
4. `GlobalAveragePooling2D()` instead of `Flatten()`

`Flatten()` can create many parameters and increase overfitting.  
`GlobalAveragePooling2D()` reduces parameters and improves generalization.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l2

model = Sequential([
    Conv2D(
        32, (3, 3), activation="relu", padding="same",
        kernel_regularizer=l2(0.001),
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    ),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    Conv2D(
        64, (3, 3), activation="relu", padding="same",
        kernel_regularizer=l2(0.001)
    ),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.30),

    Conv2D(
        128, (3, 3), activation="relu", padding="same",
        kernel_regularizer=l2(0.001)
    ),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.35),

    Conv2D(
        256, (3, 3), activation="relu", padding="same",
        kernel_regularizer=l2(0.001)
    ),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.40),

    GlobalAveragePooling2D(),

    Dense(128, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.50),

    Dense(1, activation="sigmoid")
])

model.summary()

# Step 10: Compile Model

We use a smaller learning rate to make training more stable.


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Step 11: Train Model with Callbacks

## Callback Explanation

### EarlyStopping
Stops training if validation loss does not improve.

### ReduceLROnPlateau
Reduces learning rate when validation loss stops improving.

### ModelCheckpoint
Saves the best model based on validation accuracy.


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7
    ),

    ModelCheckpoint(
        "/content/best_cat_dog_basic_cnn_overfitting_solution.h5",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    )
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=callbacks
)

# Step 12: Plot Accuracy Graph

## How to Read This Graph

If training accuracy keeps increasing but validation accuracy stays low, the model is overfitting.

A better model has training and validation accuracy close to each other.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

# Step 13: Plot Loss Graph

## How to Read This Graph

If training loss decreases but validation loss increases, the model is overfitting.

A better model has both losses decreasing or staying close.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# Step 14: Evaluate Best Model

In [ ]:
loss, accuracy = model.evaluate(val_data)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

# Step 15: Load Best Saved Model

Use this cell if you want to load the best saved model later.


In [ ]:
from tensorflow.keras.models import load_model

best_model = load_model("/content/best_cat_dog_basic_cnn_overfitting_solution.h5")

print("Best model loaded successfully.")

# Step 16: Upload One Image and Predict

This cell lets the user upload any cat/dog image and get prediction.


In [ ]:
from google.colab import files
from tensorflow.keras.preprocessing import image
import numpy as np

uploaded = files.upload()

for file_name in uploaded.keys():
    img_path = file_name

    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = best_model.predict(img_array)

    plt.imshow(img)
    plt.axis("off")
    plt.show()

    if prediction[0][0] > 0.5:
        print("Prediction: Dog")
        print("Confidence:", float(prediction[0][0]))
    else:
        print("Prediction: Cat")
        print("Confidence:", float(1 - prediction[0][0]))

# Final Notes for Students

## What We Improved

The previous basic CNN may overfit because it memorizes training images.

This notebook reduces overfitting using:

- Data augmentation
- Dropout
- Batch Normalization
- L2 Regularization
- Global Average Pooling
- EarlyStopping
- Learning rate reduction

## Expected Result

For a basic CNN, a good result is:

| Metric | Expected Range |
|---|---|
| Training Accuracy | 80% to 90% |
| Validation Accuracy | 80% to 88% |

Transfer learning models such as MobileNet, EfficientNet, or ResNet can perform better, but this notebook is designed for beginner-level CNN learning.
